# Časť 2, Téma 3: Výpis pamäte pomocou Voltage Glitchingu (MAIN)

---
POZNÁMKA: Toto praktické cvičenie odkazuje na niektoré (komerčné) školiace materiály na webe [ChipWhisperer.io](https://www.ChipWhisperer.io). Cvičenie môžete voľne spustiť a používať v súlade s licenciou open-source (vrátane jeho použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), musíte však zachovať informáciu o tomto pôvodnom zdroji. Ak si chcete vyskúšať plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *V predchádzajúcich cvičeniach sme sa naučili, že vkladanie chýb do napájania (voltage glitching) slúži na podobný účel ako hodinové glitchovanie (clock glitching). Zistili sme tiež, že má menej obmedzení, no pri určitých konfiguráciách cieľového zariadenia môže byť menej spoľahlivé. Jeho správanie sa výrazne mení v závislosti od vlastností samotného glitchovacieho obvodu – dokonca aj výmena jediného vodiča môže mať obrovský vplyv.*

*V tomto cvičení využijeme nadobudnuté vedomosti z predchádzajúcej úlohy a znova zaútočíme na zraniteľný sériový výpis bootloadera.*

**VÝSLEDKY VZDELÁVANIA:**

* Aplikovanie predchádzajúcich nastavení glitchovania na nový firmvér
* Kontrola úspešnosti a zlyhania pri vykonávaní glitchu
* Pochopenie toho, ako môžu optimalizácie kompilátora spôsobiť neočakávané a zvláštne správanie zariadení

## Kontext situácie

Kontext situácie by ste už mali dobre poznať z vašich predchádzajúcich pokusov o glitchovanie tohto bootloadera (rovnako ako aj samotnú zraniteľnosť). Na začiatku slučky nemusíte zdĺhavo vyhľadávať nové parametre, stačí použiť hodnoty, ktoré fungovali v predchádzajúcom tútoriáli.

Dávajte pozor, aby ste omylom neumiestnili cieľové miesto pre glitch mimo rozsah `glitch_spots` – ak ste použili hodnotu repeat > 1, skutočné miesto, kde nastane chyba, sa môže nachádzať na konci alebo uprostred sekvencie opakovania!

Podobne ako pri verzii tohto cvičenia s hodinovým glitchovaním, aj tu použijeme protokol SimpleSerial V2 na zrýchlenie celého procesu.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../firmware/mcu/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [ ]:
cw.program_target(scope, prog, fw_path)

Pri hardvéri ChipWhisperer (okrem verzie Husky) znova použijeme vyššiu frekvenciu. Taktiež využijeme dĺžku trvania triggeru na určenie rozsahu pre `ext_offset`:

In [ ]:
scope.clock.adc_src = "clkgen_x1"
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
    
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

if PLATFORM == "CWLITEXMEGA":
    scope.clock.clkgen_freq = 32E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*32/7.37
    else:
        target.baud = 38400*32/7.37
elif (PLATFORM == "CWLITEARM") or ("F3" in PLATFORM):
    scope.clock.clkgen_freq = 24E6
    if SS_VER=='SS_VER_2_1':
        target.baud = 230400*24/7.37
    else:
        target.baud = 38400*24/7.37

reboot_flush()
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
        
trig_count = scope.adc.trig_count

print(trig_count)
cw.plot(scope.get_last_trace())

Rovnako ako pri verzii tohto cvičenia s hodinami bude vašou úlohou analyzovať výkonovú stopu a aplikovať glitch blízko začiatku a konca slučky.

In [ ]:
glitch_spots = [i for i in range(1)]
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")
# ###################

In [ ]:
scope.vglitch_setup('both')

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="width", y_index="ext_offset")

Teraz je už zvyšok na vás!

In [ ]:
#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)
scope.adc.timeout = 0.2

gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step([???])
gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])

gc.set_range("tries", 1, 1) # change this if you want to glitch each spot multiple times
gc.set_step("tries", 1)

broken = False
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    if broken:
        break
    scope.glitch.ext_offset = glitch_setting[2]
    if scope.adc.state:
        #print("Timeout, trigger still high!")
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if ??? in output:
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = True 
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            break
        else:
            gc.add("normal")
                
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

In [ ]:
scope.dis()
target.dis()